Generate bike counting station datasets

Berlin

In [25]:
import pandas as pd
import geopandas as gpd

station_info_df = pd.read_excel("stations_datasets/raw/gesamtdatei-stundenwerte.xlsx", sheet_name="Standortdaten").rename(columns={
    'Zählstelle': 'counting_station', 'Beschreibung - Fahrtrichtung': 'address', 'Breitengrad': 'latitude', 'Längengrad': 'longitude',
    'Installationsdatum': 'install_date'
})

station_info_df["station_id"] = "BER-" + station_info_df["counting_station"]

station_info_ref_gdf = gpd.GeoDataFrame(
    station_info_df, geometry=gpd.points_from_xy(station_info_df.longitude, station_info_df.latitude), crs="EPSG:4326"
)[["station_id","geometry"]].rename(columns={"station_id": "id"})
station_info_ref_gdf["fetched_city"] = "Berlin"
station_info_ref_gdf.to_file("stations_datasets/processed/berlin_stations_ref.json", index=False)
station_info_ref_gdf

,id,geometry,fetched_city
0,BER-12-PA-SCH,POINT (13.40037 52.54907),Berlin
1,BER-02-MI-JAN-N,POINT (13.41783 52.51393),Berlin
2,BER-02-MI-JAN-S,POINT (13.41761 52.51394),Berlin
3,BER-13-CW-PRI,POINT (13.33312 52.48814),Berlin
4,BER-18-TS-YOR-O,POINT (13.37347 52.49194),Berlin
5,BER-18-TS-YOR-W,POINT (13.37321 52.49228),Berlin
6,BER-19-TS-MON,POINT (13.36978 52.48812),Berlin
7,BER-27-RE-MAR,POINT (13.36494 52.55819),Berlin
8,BER-03-MI-SAN-O,POINT (13.37202 52.52718),Berlin
9,BER-03-MI-SAN-W,POINT (13.3731 52.52769),Berlin


In [26]:
berlin_year_2024_df = pd.read_excel("stations_datasets/raw/gesamtdatei-stundenwerte.xlsx", sheet_name="Jahresdatei 2024").rename(columns=
                                                                                                                                 {'Zählstelle        Inbetriebnahme': 'datetime'})
berlin_year_2024_df

,datetime,01-MI-AL-W\n16.12.2021,02-MI-JAN-N\n01.04.2015,02-MI-JAN-S\n01.04.2015,03-MI-SAN-O\n01.06.2015,03-MI-SAN-W\n01.06.2015,04-MI-NO\n16.10.2023,05-FK-OBB-O\n01.06.2015,05-FK-OBB-W\n01.06.2015,06-FK-FRA-O\n01.06.2016,...,18-TS-YOR-O\n01.04.2015,18-TS-YOR-W\n01.04.2015,19-TS-MON\n01.05.2015,20-TS-MAR-N\n01.05.2016,20-TS-MAR-S\n01.05.2016,21-NK-MAY\n01.05.2016,23-TK-KAI\n01.05.2016,24-MH-ALB\n01.07.2015,26-LI-PUP\n01.06.2015,27-RE-MAR\n01.05.2015
0,2024-01-01 00:00:00,5.0,8.0,9.0,18.0,4.0,2.0,20.0,21.0,6.0,...,8.0,7.0,6.0,0.0,1.0,15.0,4.0,0.0,8.0,0.0
1,2024-01-01 01:00:00,11.0,18.0,18.0,27.0,9.0,3.0,48.0,24.0,12.0,...,21.0,12.0,26.0,3.0,2.0,25.0,2.0,6.0,8.0,4.0
2,2024-01-01 02:00:00,15.0,28.0,25.0,17.0,8.0,4.0,55.0,42.0,17.0,...,28.0,21.0,30.0,1.0,3.0,31.0,12.0,4.0,28.0,7.0
3,2024-01-01 03:00:00,13.0,21.0,8.0,10.0,7.0,1.0,44.0,33.0,13.0,...,13.0,16.0,13.0,3.0,2.0,53.0,12.0,7.0,15.0,6.0
4,2024-01-01 04:00:00,5.0,18.0,19.0,10.0,10.0,3.0,38.0,34.0,5.0,...,9.0,7.0,13.0,3.0,2.0,25.0,6.0,3.0,10.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8779,2024-12-31 19:00:00,28.0,33.0,2.0,30.0,26.0,10.0,94.0,73.0,29.0,...,36.0,38.0,37.0,10.0,3.0,91.0,27.0,9.0,37.0,13.0
8780,2024-12-31 20:00:00,21.0,31.0,1.0,41.0,23.0,5.0,51.0,51.0,18.0,...,18.0,23.0,40.0,8.0,5.0,51.0,10.0,2.0,15.0,7.0
8781,2024-12-31 21:00:00,6.0,27.0,0.0,17.0,5.0,4.0,42.0,46.0,15.0,...,14.0,6.0,19.0,4.0,2.0,49.0,11.0,5.0,13.0,5.0
8782,2024-12-31 22:00:00,5.0,22.0,1.0,12.0,21.0,5.0,46.0,46.0,10.0,...,14.0,7.0,12.0,2.0,1.0,19.0,4.0,5.0,9.0,7.0


In [ ]:
import re

def rename_berlin_columns(col):
    if col == 'datetime':
        return col
    
    return "BER-" + re.split(pattern="\n",string=col)[0]
berlin_year_2024_df = berlin_year_2024_df.rename(rename_berlin_columns,axis="columns")


In [23]:
count_cols = berlin_year_2024_df.filter(regex="BER").columns
berlin_year_2024_df_long = berlin_year_2024_df.melt(
    id_vars="datetime",
    value_vars=count_cols,
    var_name="id",
    value_name="count"
).rename(columns={"datetime": "time"})
berlin_year_2024_df_long

,time,id,count
0,2024-01-01 00:00:00,BER-01-MI-AL-W,5.0
1,2024-01-01 01:00:00,BER-01-MI-AL-W,11.0
2,2024-01-01 02:00:00,BER-01-MI-AL-W,15.0
3,2024-01-01 03:00:00,BER-01-MI-AL-W,13.0
4,2024-01-01 04:00:00,BER-01-MI-AL-W,5.0
...,...,...,...
307435,2024-12-31 19:00:00,BER-27-RE-MAR,13.0
307436,2024-12-31 20:00:00,BER-27-RE-MAR,7.0
307437,2024-12-31 21:00:00,BER-27-RE-MAR,5.0
307438,2024-12-31 22:00:00,BER-27-RE-MAR,7.0


In [24]:
berlin_year_2024_df_long.to_csv("stations_datasets/processed/berlin_stations_count.csv", index=False)

In [ ]:
# station_id_list = station_info_df["station_id"].unique()
# daily_grouped_berlin_2024_df = berlin_year_2024_df.groupby(by=berlin_year_2024_df["datetime"].dt.date).sum(numeric_only=True).reset_index()
# daily_grouped_berlin_2024_df

,datetime,BER-01-MI-AL-W,BER-02-MI-JAN-N,BER-02-MI-JAN-S,BER-03-MI-SAN-O,BER-03-MI-SAN-W,BER-04-MI-NO,BER-05-FK-OBB-O,BER-05-FK-OBB-W,BER-06-FK-FRA-O,...,BER-18-TS-YOR-O,BER-18-TS-YOR-W,BER-19-TS-MON,BER-20-TS-MAR-N,BER-20-TS-MAR-S,BER-21-NK-MAY,BER-23-TK-KAI,BER-24-MH-ALB,BER-26-LI-PUP,BER-27-RE-MAR
0,2024-01-01,459.0,608.0,513.0,434.0,389.0,222.0,1287.0,1218.0,475.0,...,587.0,507.0,591.0,161.0,148.0,875.0,310.0,105.0,580.0,319.0
1,2024-01-02,1107.0,1360.0,1348.0,811.0,842.0,445.0,1943.0,1893.0,1075.0,...,1056.0,941.0,1296.0,438.0,386.0,1950.0,860.0,158.0,981.0,961.0
2,2024-01-03,1593.0,2098.0,2051.0,1126.0,1193.0,586.0,3004.0,2837.0,1416.0,...,1525.0,1435.0,1912.0,555.0,475.0,2723.0,1156.0,245.0,1531.0,1056.0
3,2024-01-04,1369.0,1929.0,1963.0,1087.0,1093.0,544.0,2846.0,2657.0,1229.0,...,1394.0,1302.0,1759.0,458.0,413.0,2639.0,1085.0,201.0,1369.0,1051.0
4,2024-01-05,947.0,1340.0,1206.0,802.0,785.0,392.0,2186.0,2010.0,942.0,...,1004.0,918.0,1277.0,402.0,329.0,2217.0,821.0,124.0,866.0,834.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,959.0,1153.0,65.0,625.0,629.0,424.0,1916.0,1899.0,992.0,...,1030.0,932.0,1108.0,378.0,331.0,1684.0,612.0,225.0,738.0,620.0
362,2024-12-28,793.0,941.0,51.0,537.0,532.0,320.0,1877.0,1831.0,844.0,...,986.0,893.0,1007.0,319.0,229.0,1735.0,505.0,150.0,619.0,523.0
363,2024-12-29,592.0,744.0,51.0,493.0,537.0,258.0,1853.0,1787.0,676.0,...,721.0,573.0,763.0,190.0,165.0,1211.0,458.0,100.0,495.0,391.0
364,2024-12-30,1040.0,1248.0,80.0,808.0,704.0,445.0,2238.0,2243.0,1128.0,...,1258.0,1136.0,1344.0,428.0,385.0,2242.0,663.0,181.0,792.0,671.0


datetime
1      45142.0
2      61028.0
3      81182.0
4      91787.0
5     115579.0
6     111605.0
7     120303.0
8     116983.0
9     105009.0
10     91180.0
11     71194.0
12     49112.0
Name: 01, dtype: float64

In [19]:
berlin_year_2024_df.index[1]


1

## Hamburg

In [1]:
import requests

url = "https://iot.hamburg.de/v1.1/Datastreams"
params = {
    "$filter": (
        "properties/serviceName eq 'HH_STA_HamburgerRadzaehlnetz' and "
        "properties/layerName eq 'Anzahl_Fahrraeder_Zaehlstelle_1-Tag'"
    ),
    "$expand": "Thing($select=@iot.id)",
    "$select": "@iot.id"
}


r = requests.get(url, params=params)
r.raise_for_status()
datastreams = r.json()["value"]


In [2]:
def fetch_all_observations(datastream_id):
    url = f"https://iot.hamburg.de/v1.1/Datastreams({datastream_id})/Observations"
    params = {
        "$orderby": "phenomenonTime asc",
        "$top": 1000
    }

    all_obs = []

    while url:
        r = requests.get(url, params=params)
        r.raise_for_status()
        payload = r.json()

        all_obs.extend(payload["value"])
        url = payload.get("@iot.nextLink")
        params = None 

    return all_obs


In [3]:
import pandas as pd

rows = []

for ds in datastreams:
    ds_id = ds["@iot.id"]
    thing_id = ds["Thing"]["@iot.id"]

    observations = fetch_all_observations(ds_id)

    for obs in observations:
        date = pd.to_datetime(
            obs["phenomenonTime"].split("/")[1]
        ).date()

        rows.append({
            "sensor_id": thing_id,
            "datastream_id": ds_id,
            "date": date,
            "bike_count": obs["result"]
        })


In [4]:

df = pd.DataFrame(rows)

df


,sensor_id,datastream_id,date,bike_count
0,5564,11798,2020-07-28,8868
1,5564,11798,2020-07-29,8310
2,5564,11798,2020-07-30,9486
3,5564,11798,2020-07-31,9750
4,5564,11798,2020-08-01,6263
...,...,...,...,...
202891,5883,12831,2026-02-25,4768
202892,5883,12831,2026-02-26,4673
202893,5883,12831,2026-02-27,6204
202894,5883,12831,2026-02-28,2820


In [ ]:
df["date"] = pd.to_datetime(df["date"])
df_2025 = df[df["date"].dt.year == 2025].rename(columns={"sensor_id":"id", "bike_count":"count","date":"time"}).drop(columns=["datastream_id"])
df_2025["id"] = 'HAM_' + (df_2025["id"]).astype(str)
df_2025["fetched_city"] = "Hamburg"


,id,datastream_id,time,count
1592,HAM_5564,11798,2025-01-01,658
1593,HAM_5564,11798,2025-01-02,2706
1594,HAM_5564,11798,2025-01-03,1469
1595,HAM_5564,11798,2025-01-04,959
1596,HAM_5564,11798,2025-01-05,1010
...,...,...,...,...
202831,HAM_5883,12831,2025-12-27,866
202832,HAM_5883,12831,2025-12-28,835
202833,HAM_5883,12831,2025-12-29,1635
202834,HAM_5883,12831,2025-12-30,1705


In [8]:
df_2025.to_csv("stations_datasets/processed/hamburg_stations_count.csv",index=False)

In [19]:
df_2025

,sensor_id,datastream_id,date,bike_count
1592,5564,11798,2025-01-01,658
1593,5564,11798,2025-01-02,2706
1594,5564,11798,2025-01-03,1469
1595,5564,11798,2025-01-04,959
1596,5564,11798,2025-01-05,1010
...,...,...,...,...
202916,5884,12835,2025-12-27,474
202917,5884,12835,2025-12-28,484
202918,5884,12835,2025-12-29,870
202919,5884,12835,2025-12-30,925


In [33]:
df[df["sensor_id"] == 5564]

,sensor_id,datastream_id,date,bike_count
0,5564,11798,2020-07-28,8868
1,5564,11798,2020-07-29,8310
2,5564,11798,2020-07-30,9486
3,5564,11798,2020-07-31,9750
4,5564,11798,2020-08-01,6263
...,...,...,...,...
1817,5564,11798,2025-08-14,10109
1818,5564,11798,2025-08-15,8738
1819,5564,11798,2025-08-16,6430
1820,5564,11798,2025-08-17,7497


In [26]:
obs_per_day = (
    df
    .groupby(["sensor_id", "date"])
    .size()
    .reset_index(name="n_observations")
)


In [27]:
obs_per_day

,sensor_id,date,n_observations
0,5564,2020-07-28,1
1,5564,2020-07-29,1
2,5564,2020-07-30,1
3,5564,2020-07-31,1
4,5564,2020-08-01,1
...,...,...,...
199877,5884,2026-01-24,1
199878,5884,2026-01-25,1
199879,5884,2026-01-26,1
199880,5884,2026-01-27,1


In [14]:
df

,sensor_id,datastream_id,phenomenonTime,result
0,5885,12839,2026-01-27T23:00:00Z/2026-01-28T22:59:59Z,829
1,5885,12839,2026-01-26T23:00:00Z/2026-01-27T22:59:59Z,621
2,5885,12839,2026-01-25T23:00:00Z/2026-01-26T22:59:59Z,332
3,5885,12839,2026-01-24T23:00:00Z/2026-01-25T22:59:59Z,215
4,5885,12839,2026-01-23T23:00:00Z/2026-01-24T22:59:59Z,354
...,...,...,...,...
995,9218,19872,2026-01-22T23:00:00Z/2026-01-23T22:59:59Z,604
996,9218,19872,2026-01-21T23:00:00Z/2026-01-22T22:59:59Z,886
997,9218,19872,2026-01-20T23:00:00Z/2026-01-21T22:59:59Z,824
998,9218,19872,2026-01-19T23:00:00Z/2026-01-20T22:59:59Z,949


In [9]:
import requests

def fetch_all_things():
    url = "https://iot.hamburg.de/v1.1/Things"
    params = {
        "$filter": "Datastreams/properties/serviceName eq 'HH_STA_HamburgerRadzaehlnetz'",
        "$expand": "Locations",
        "$count": "true"
    }

    all_things = []

    while url:
        r = requests.get(url, params=params)
        r.raise_for_status()
        payload = r.json()

        all_things.extend(payload["value"])
        url = payload.get("@iot.nextLink")
        params = None  # nextLink already includes params

    return all_things


things = fetch_all_things()
len(things)  # should be ~670


670

In [10]:
rows = []

for thing in things:
    if not thing.get("Locations"):
        continue

    coords = thing["Locations"][0]["location"]["geometry"]["coordinates"]

    rows.append({
        "sensor_id": thing["@iot.id"],
        "name": thing["name"],
        "asset_id": thing["properties"].get("assetID"),
        "lon": coords[0],
        "lat": coords[1]
    })


In [11]:
import geopandas as gpd
from shapely.geometry import Point

gdf_stations = gpd.GeoDataFrame(
    rows,
    geometry=[Point(xy) for xy in zip(
        [r["lon"] for r in rows],
        [r["lat"] for r in rows]
    )],
    crs="EPSG:4326"
)

gdf_stations


,sensor_id,name,asset_id,lon,lat,geometry
0,5564,Verkehrszählstelle 0295970 (veraltet),0295970,9.999098,53.579928,POINT (9.9991 53.57993)
1,5565,Verkehrszählstelle 0295971 (veraltet),0295971,9.998994,53.579992,POINT (9.99899 53.57999)
2,5566,Verkehrszählstelle 0295972 (veraltet),0295972,9.999202,53.579864,POINT (9.9992 53.57986)
3,5570,Verkehrszählstelle 0295910 (veraltet),0295910,9.999531,53.580518,POINT (9.99953 53.58052)
4,5571,Verkehrszählstelle 0295911 (veraltet),0295911,9.999510,53.580504,POINT (9.99951 53.5805)
...,...,...,...,...,...,...
665,11893,Verkehrszählstelle 0307930 (veraltet),0307930,10.018337,53.564655,POINT (10.01834 53.56466)
666,11894,Verkehrszählstelle 0307931 (veraltet),0307931,10.018233,53.564719,POINT (10.01823 53.56472)
667,11895,Verkehrszählstelle 0307932 (veraltet),0307932,10.018441,53.564592,POINT (10.01844 53.56459)
668,11964,Zählfeld J_87.1_1_I (veraltet),J_87.1_1_I,9.912326,53.560310,POINT (9.91233 53.56031)


In [ ]:
gdf_stations_final = gdf_stations[["sensor_id","geometry"]].rename(columns={"sensor_id":"id"})
gdf_stations_final["id"] = "HAM_" + gdf_stations_final["id"].astype(str)
gdf_stations_final["fetched_city"] = "Hamburg"



In [28]:
gdf_stations_final_2025 = gdf_stations_final.merge(df_2025["id"], how="inner").drop_duplicates()
gdf_stations_final_2025.to_file("stations_datasets/processed/hamburg_stations_ref.json", index=False)

In [15]:
gdf_stations[gdf_stations["sensor_id"] == 9391]

,sensor_id,name,asset_id,lon,lat,geometry
530,9391,Zählfeld D_26.2_1_I (veraltet),D_26.2_1_I,9.989417,53.561417,POINT (9.98942 53.56142)


In [21]:
df_2025[df_2025["id"] == 9397]

,id,datastream_id,time,count


In [30]:
gdf_stations_final_2025_counts = df_2025.merge(gdf_stations_final[["id","geometry"]], how="inner").drop_duplicates().drop(columns=["datastream_id"])
gdf_stations_final_2025_counts

,id,time,count,geometry
0,HAM_5564,2025-01-01,658,POINT (9.9991 53.57993)
1,HAM_5564,2025-01-02,2706,POINT (9.9991 53.57993)
2,HAM_5564,2025-01-03,1469,POINT (9.9991 53.57993)
3,HAM_5564,2025-01-04,959,POINT (9.9991 53.57993)
4,HAM_5564,2025-01-05,1010,POINT (9.9991 53.57993)
...,...,...,...,...
36018,HAM_5883,2025-12-27,866,POINT (9.99576 53.55938)
36019,HAM_5883,2025-12-28,835,POINT (9.99576 53.55938)
36020,HAM_5883,2025-12-29,1635,POINT (9.99576 53.55938)
36021,HAM_5883,2025-12-30,1705,POINT (9.99576 53.55938)


In [34]:
gdf_stations_final_2025_grouped = gpd.GeoDataFrame(gdf_stations_final_2025_counts[["id","geometry","count"]].groupby(["id","geometry"], as_index=False).sum())
gdf_stations_final_2025_grouped

,id,geometry,count
0,HAM_5564,POINT (9.9991 53.57993),1769433
1,HAM_5565,POINT (9.99899 53.57999),1081337
2,HAM_5566,POINT (9.9992 53.57986),688096
3,HAM_5570,POINT (9.99953 53.58052),951915
4,HAM_5571,POINT (9.99951 53.5805),544121
...,...,...,...
95,HAM_5880,POINT (10.04063 53.59835),205988
96,HAM_5881,POINT (9.9871 53.457),317228
97,HAM_5882,POINT (9.98708 53.45706),163684
98,HAM_5883,POINT (9.99576 53.55938),1512093


In [32]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

DB_CONFIG = {
    'host': 'localhost',
    'dbname': os.getenv("POSTGRES_DB"),
    'user': os.getenv("POSTGRES_USER"),
    'password': os.getenv("POSTGRES_PASSWORD"),
    'port': 5432
}
engine = create_engine(f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}@"
                f"{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")
load_dotenv()



True

In [35]:
gdf_stations_final_2025_grouped.to_postgis("counting_stations_hamburg",engine)

In [45]:
stations_spatial_merge = gpd.GeoDataFrame.from_file("stations_datasets/processed/hamburg_stations_spatial_merge.geojson")
stations_spatial_merge

,id,geometry
0,1,"MULTILINESTRING ((9.96639 53.54083, 9.96662 53..."
1,2,"MULTILINESTRING ((9.93399 53.55036, 9.93402 53..."
2,6,"MULTILINESTRING ((9.98975 53.55962, 9.99019 53..."
3,7,"MULTILINESTRING ((9.99572 53.55938, 9.99604 53..."
4,21,"MULTILINESTRING ((9.99797 53.55786, 9.9978 53...."
5,8,"MULTILINESTRING ((10.01994 53.56622, 10.01995 ..."
6,9,"MULTILINESTRING ((9.99897 53.58, 9.99923 53.57..."
7,10,"MULTILINESTRING ((9.99957 53.58057, 9.99944 53..."
8,11,"MULTILINESTRING ((9.97072 53.58164, 9.971 53.5..."
9,12,"MULTILINESTRING ((9.9728 53.58151, 9.97237 53...."


In [46]:
stations_spatial_merge.to_postgis("counting_stations_lines_hamburg",engine, index=False, if_exists="replace")